In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from loss import PDE_loss, dirichlet_loss, all_at_once_loss
from utils import get_knots
from model import PINN_Net, PINN_Net_Fourier, FourierFeatureLayer, ScaledTanh, PINN_Net_Adapt, P2INN
from data_prep import sample_collocation, sample_terminal, AdaptivePDEDataset, PDEDataset, TerminalDataset, AdaptiveTerminalDataset, AdaptiveTerminalDataset1, PINNDataFactory
from torch.utils.data import DataLoader
from utils import batch_get_difference, grad_net, Simpson_rule, trapz_rule, grad_log_w, simulate_samples, MichaelisMentenDrift, diff_function
import argparse
import os
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
from itertools import count
from sklearn.linear_model import LinearRegression, Ridge

In [2]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

In [3]:
T = 0.5
# K = 2
n = 500
dt = T / n
t_0 = 0.0
t_grid = get_knots(T, n)
k1 = 1.3634
k_minus1 = 2.4379
k2 = 0.2018
sigma = 0.3
# mu = 0
# sigma = 0.5
r = 0.2
x_min = 0.0
x_max = 2.0
batch_size = 1000
n_points = 101
x0 = torch.tensor([1.0, 1.0], device="cpu")
d = x0.shape[-1]
drift_func = MichaelisMentenDrift(k1, k_minus1, k2, 3)
diff_func = diff_function(sigma)
Sigma = diff_func(x0)

print(Sigma)


tensor([0.3000, 0.3000])


In [4]:
_, X, Y = simulate_samples(MichaelisMentenDrift(1.0, 1.5, 1.0, 3), diff_func, T, n, -10.0, 10.0, X0=x0, seed=1007)
obs_times = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
obs_idx = [t / dt for t in obs_times]
x = X[:, obs_idx]
y = Y[:, obs_idx]
N = len(obs_times) - 1
print(y)
print(N)
print(X)

tensor([[[1.1533, 0.8633],
         [1.4251, 1.0022],
         [1.0568, 1.4031],
         [0.9197, 1.7100],
         [0.9116, 1.7211],
         [1.4645, 1.8734]]], device='mps:0')
5
tensor([[[1.0000, 1.0000],
         [1.0018, 1.0229],
         [0.9997, 1.0302],
         ...,
         [1.1864, 1.8359],
         [1.1958, 1.8289],
         [1.2013, 1.8301]]], device='mps:0')


In [5]:
models = nn.ModuleList([PINN_Net(d, 5, 70) for _ in range(N+1)]).to(device)

In [6]:
data_factory = PINNDataFactory(obs_times, x_min, x_max, d, 1000, 1000, dtype=torch.float32, device="cpu")

In [7]:
ckpt_path = "Michaelis_Menten_5_J3.0_2.4074_3.0769_0.8245_xmin0.0_max2.5_all_at_once_diff0.3_initial_[1.1533, 0.8633]/pinn_models_all_at_once.pt"
state = torch.load(ckpt_path, map_location="cpu")
models.load_state_dict(state["models_state_dict"])
models.to(device)

/var/folders/sy/w43hjzh55551685cq4p3x5v00000gn/T/ipykernel_21154/2102510147.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt_path, map_location="c

ModuleList(
  (0-5): 6 x PINN_Net(
    (net): Sequential(
      (0): Linear(in_features=3, out_features=70, bias=True)
      (1): Tanh()
      (2): Sequential(
        (0): Linear(in_features=70, out_features=70, bias=True)
        (1): Tanh()
      )
      (3): Sequential(
        (0): Linear(in_features=70, out_features=70, bias=True)
        (1): Tanh()
      )
      (4): Sequential(
        (0): Linear(in_features=70, out_features=70, bias=True)
        (1): Tanh()
      )
      (5): Sequential(
        (0): Linear(in_features=70, out_features=70, bias=True)
        (1): Tanh()
      )
      (6): Sequential(
        (0): Linear(in_features=70, out_features=70, bias=True)
        (1): Tanh()
      )
      (7): Sequential(
        (0): Linear(in_features=70, out_features=1, bias=True)
      )
    )
  )
)

In [8]:
# Train all the PINNs at the same time via all-at-once loss and data factory 
def PINN_train(nets, drift_func, diff_func, data_factory, early_stop_threshold, not_in_EM=True):
    optimizer = torch.optim.AdamW(nets.parameters(), lr=1e-4)
    consecutive_below = 0

    for epoch in count(0):
        for net in nets:
            net.train()

        optimizer.zero_grad(set_to_none=True)
        interval_loaders, knot_loaders = data_factory()
        loss = all_at_once_loss(interval_loaders, knot_loaders, nets, drift_func, diff_func, y=y, r=r)
        loss.backward()
        optimizer.step()

        loss_val = float(loss.detach().cpu())
        if not_in_EM:
            print(f"epoch {epoch}: loss = {loss_val:.4f}")

        if loss_val < early_stop_threshold:
            consecutive_below += 1
        else:
            consecutive_below = 0   

        if consecutive_below > 3:
            break   

In [9]:
PINN_train(models, drift_func, diff_func, data_factory, early_stop_threshold=20.0)

epoch 0: loss = 7613.8882
epoch 1: loss = 6667.1958
epoch 2: loss = 6184.8989
epoch 3: loss = 5688.2905
epoch 4: loss = 5015.0137
epoch 5: loss = 4201.9170
epoch 6: loss = 3923.6011
epoch 7: loss = 3188.0967
epoch 8: loss = 3187.2305
epoch 9: loss = 2926.9795
epoch 10: loss = 2677.9167
epoch 11: loss = 2291.1899
epoch 12: loss = 2120.3706
epoch 13: loss = 1834.0195
epoch 14: loss = 1749.7682
epoch 15: loss = 1646.7180
epoch 16: loss = 1557.9333
epoch 17: loss = 1368.8920
epoch 18: loss = 1349.9312
epoch 19: loss = 1197.8815
epoch 20: loss = 1175.0410
epoch 21: loss = 1121.9265
epoch 22: loss = 1003.9103
epoch 23: loss = 961.6298
epoch 24: loss = 923.8289
epoch 25: loss = 883.0328
epoch 26: loss = 822.3522
epoch 27: loss = 733.9717
epoch 28: loss = 736.2604
epoch 29: loss = 713.2065
epoch 30: loss = 699.0879
epoch 31: loss = 647.8486
epoch 32: loss = 642.2007
epoch 33: loss = 615.9793
epoch 34: loss = 583.1464
epoch 35: loss = 563.4070
epoch 36: loss = 565.0444
epoch 37: loss = 533.7469

In [ ]:
save_dir = "Michaelis_Menten_5_J3.0_2.4074_3.0769_0.8245_xmin0.0_max2.5_all_at_once_diff0.3_initial_[1.1533, 0.8633]"
os.makedirs(save_dir, exist_ok=True)
ckpt_path = os.path.join(save_dir, f"pinn_models_all_at_once.pt")

ckpt = {
    "models_state_dict": models.state_dict(),  # works for ModuleList (recursive)
    # (optional) include anything else you want to remember:
    "meta": {"N": N, "in_dim": 1, "out_dim": 5, "hidden": 70}
}
torch.save(ckpt, ckpt_path)
print(f"Saved: {ckpt_path}")

In [10]:
# def mle_kappa(x, dt, sigma, ridge=0.0):
#     """
#     Closed-form weighted least squares M-step for theta = (k1, k_minus1, k2)
#     using Euler Gaussian increments.

#     Parameters
#     ----------
#     x : torch.Tensor, shape (M, T, 3)
#         Simulated trajectories (latent paths) in EM E-step.
#     dt : float
#         Time step (assumed constant).
#     sigma : torch.Tensor or array-like, shape (3,)
#         Constant diagonal diffusion entries (sigma1, sigma2, sigma3).
#     ridge : float
#         Optional L2 regularization (adds ridge*I to X^T W X).

#     Returns
#     -------
#     theta : torch.Tensor, shape (3,)
#         [k1, k_minus1, k2]
#     """
#     if x.shape[-1] != 3:
#         raise ValueError(f"x must have last dim=3; got {tuple(x.shape)}")
#     M, T, d = x.shape
#     if T < 2:
#         raise ValueError("Need at least two time points to form increments.")

#     sigma = torch.as_tensor(sigma, dtype=x.dtype, device=x.device)
#     if sigma.shape != (3,):
#         raise ValueError(f"sigma must be (3,), got {tuple(sigma.shape)}")

#     # Increments and "targets" y = dx/dt
#     x_n   = x[:, :-1, :]              # (M, T-1, 3)
#     dx    = x[:, 1:, :] - x_n         # (M, T-1, 3)
#     y     = (dx / dt).reshape(-1, 3)  # (N, 3), where N = M*(T-1)

#     # Features
#     phi1 = (x_n[..., 0] * x_n[..., 1]).reshape(-1)  # (N,)
#     phi2 = (x_n[..., 2]).reshape(-1)                # (N,)

#     # Build design rows for each component, stack into (3N, 3)
#     # row1: [-phi1, +phi2, 0]
#     # row2: [-phi1, +phi2, +phi2]
#     # row3: [+phi1, -phi2, -phi2]
#     N = phi1.shape[0]
#     X1 = torch.stack((-phi1,  phi2, torch.zeros_like(phi1)), dim=1)
#     X2 = torch.stack((-phi1,  phi2,  phi2), dim=1)
#     X3 = torch.stack(( phi1, -phi2, -phi2), dim=1)
#     X  = torch.cat([X1, X2, X3], dim=0)  # (3N, 3)

#     # Stack targets into (3N,)
#     y_stack = torch.cat([y[:, 0], y[:, 1], y[:, 2]], dim=0)  # (3N,)

#     # Weights: w_i = 1/sigma_i^2 repeated N times each
#     w = torch.cat([
#         torch.full((N,), dt/(sigma[0]**2), dtype=x.dtype, device=x.device),
#         torch.full((N,), dt/(sigma[1]**2), dtype=x.dtype, device=x.device),
#         torch.full((N,), dt/(sigma[2]**2), dtype=x.dtype, device=x.device),
#     ], dim=0)  # (3N,)

#     # Solve weighted normal equations: (X^T W X) theta = X^T W y
#     # Implement W by multiplying rows by sqrt(w)
#     sqrtw = torch.sqrt(w).unsqueeze(1)      # (3N, 1)
#     Xw = X * sqrtw                          # (3N, 3)
#     yw = y_stack * sqrtw.squeeze(1)         # (3N,)

#     A = Xw.T @ Xw                           # (3, 3)
#     b = Xw.T @ yw                           # (3,)

#     if ridge > 0:
#         A = A + ridge * torch.eye(3, dtype=x.dtype, device=x.device)

#     theta = torch.linalg.solve(A, b)        # (3,)
#     return theta  # [k1, k_minus1, k2]

def mle_kappa(paths, dt, J):
    """
    paths: (n_paths, n_steps+1, 2)
    """

    paths = paths.detach().cpu().numpy()

    x = paths[:, :-1, :]
    x_next = paths[:, 1:, :]

    dx = (x_next - x) / dt

    x1 = x[..., 0]
    x2 = x[..., 1]

    a = x1 * x2
    b = J - x2

    y1 = dx[..., 0].reshape(-1)
    y2 = dx[..., 1].reshape(-1)

    a = a.reshape(-1)
    b = b.reshape(-1)

    # build design matrix
    A1 = np.stack([-a, b, np.zeros_like(b)], axis=1)
    A2 = np.stack([-a, b, b], axis=1)

    A = np.concatenate([A1, A2], axis=0)
    y = np.concatenate([y1, y2])

    model = LinearRegression(fit_intercept=False)
    model.fit(A, y)

    k1, k_minus1, k2 = model.coef_

    return k1, k_minus1, k2

def mle_k1_kminus1_fixed_k2_deltaX(X, dt, J, k2_fixed, alpha_ridge=1e-6):
    """
    Estimate k1 and k_minus1 with k2 fixed, using
    Delta X = dt * drift + noise
    """
    if torch.is_tensor(X):
        X = X.detach().cpu().numpy()
    if torch.is_tensor(dt):
        dt = float(dt.detach().cpu().item())

    x = X[:, :-1, :]
    x_next = X[:, 1:, :]

    dX = x_next - x

    x1 = x[..., 0]
    x2 = x[..., 1]

    a = (x1 * x2).reshape(-1)
    b = (J - x2).reshape(-1)

    y1 = dX[..., 0].reshape(-1)
    y2 = dX[..., 1].reshape(-1) - dt * k2_fixed * b

    A1 = np.stack([-dt * a, dt * b], axis=1)
    A2 = np.stack([-dt * a, dt * b], axis=1)

    A = np.concatenate([A1, A2], axis=0)
    y = np.concatenate([y1, y2], axis=0)

    model = Ridge(alpha=alpha_ridge, fit_intercept=False)
    model.fit(A, y)

    k1_hat, k_minus1_hat = model.coef_
    return float(k1_hat), float(k_minus1_hat)

In [ ]:
n1 = 100
posterior_drift_func = lambda x, t: drift_func(x, t) + 0.2 * sigma**2 * grad_log_w(x, t, models, obs_times)
ests = []
for _ in range(10):
    t_grid, X, _ = simulate_samples(posterior_drift_func, diff_func, T, n1, x_min, x_max, X0=x0, n_paths=300, clamp=False)
    dt = t_grid[1] - t_grid[0]
    dt, X = dt.to("cpu"), X.to("cpu")
    k1, k_minus1, k2 = mle_kappa(X, dt, 3)
    ests.append(k1)
ests_np = np.array(ests)
print(np.mean(ests_np))

In [11]:
def EM_algorithm(iteration, models, k1_init, k_minus1_init, k2_init, sigma, T, n, X0, num_path, alpha):
    kappa1_list = [k1_init]
    kappa_minus1_list = [k_minus1_init]
    kappa2_list = [k2_init]
    kappa1, kappa_minus1, kappa2 = k1_init, k_minus1_init, k2_init
    prior_drift_func = MichaelisMentenDrift(kappa1, kappa_minus1, kappa2, 3)

    for iter in range(iteration):
        PINN_train(models, prior_drift_func, diff_func, data_factory=data_factory, early_stop_threshold=20.00, not_in_EM=False)
        # models = [net.to("cpu") for net in models]
        posterior_drift_func = lambda x, t: prior_drift_func(x, t) + 0.1 * diff_func(x)**2 * grad_log_w(x, t, models, obs_times)
        t_grid, X, _ = simulate_samples(posterior_drift_func, diff_func, T, n, x_min, x_max, X0=X0, n_paths=num_path, clamp=True)
        dt = t_grid[1] - t_grid[0]
        dt, X = dt.to("cpu"), X.to("cpu")
        kappa1, kappa_minus1, kappa2 = mle_kappa(X, dt, 3)
        # kappa1 = (1 - alpha) * kappa1_hat + alpha * kappa1
        # kappa_minus1 = (1 - alpha) * kappa_minus1_hat + alpha * kappa_minus1
        prior_drift_func.update(k1=kappa1, k_minus1=kappa_minus1, k2=kappa2)

        kappa1_list.append(kappa1)
        kappa_minus1_list.append(kappa_minus1)
        kappa2_list.append(kappa2)
        print(f"iter {iter}: k1 is {kappa1:.4f}, k_minus1 is {kappa_minus1:.4f}, k2 is {kappa2:.4f}")
    
    return kappa1_list, kappa_minus1_list, kappa2_list

In [12]:
kappa1_list, kappa_minus1_list, kappa2_list = EM_algorithm(1000, models, 1.3634, 2.4379, 0.2018, Sigma, T, n, x0, 50, 0.5)

iter 0: k1 is 1.3312, k_minus1 is 2.4911, k2 is 0.0830
iter 1: k1 is 1.2164, k_minus1 is 2.3142, k2 is 0.1438
iter 2: k1 is 1.1376, k_minus1 is 2.3107, k2 is 0.0501
iter 3: k1 is 1.1380, k_minus1 is 2.3650, k2 is 0.0356
iter 4: k1 is 1.0780, k_minus1 is 2.2807, k2 is 0.0884
iter 5: k1 is 1.0515, k_minus1 is 2.2621, k2 is 0.0340
iter 6: k1 is 1.0380, k_minus1 is 2.2272, k2 is 0.0374
iter 7: k1 is 0.9931, k_minus1 is 2.1774, k2 is 0.0785
iter 8: k1 is 1.0129, k_minus1 is 2.2041, k2 is 0.0685
iter 9: k1 is 1.0324, k_minus1 is 2.2462, k2 is 0.0523
iter 10: k1 is 1.0867, k_minus1 is 2.3012, k2 is 0.0897
iter 11: k1 is 1.1080, k_minus1 is 2.3837, k2 is -0.0139
iter 12: k1 is 1.0817, k_minus1 is 2.4042, k2 is -0.1109
iter 13: k1 is 1.0498, k_minus1 is 2.3769, k2 is -0.1126
iter 14: k1 is 1.0692, k_minus1 is 2.4118, k2 is -0.1051
iter 15: k1 is 1.1035, k_minus1 is 2.3949, k2 is -0.0201
iter 16: k1 is 1.0103, k_minus1 is 2.2438, k2 is -0.0272
iter 17: k1 is 0.9914, k_minus1 is 2.3016, k2 is -0.

RuntimeError: MPS backend out of memory (MPS allocated: 9.07 GB, other allocations: 768.00 KB, max allowed: 9.07 GB). Tried to allocate 273.50 KB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).